# Rechunk Without Shuffling

When working with large dask-backed rasters, rechunking to bigger blocks can
speed up downstream operations like `slope()` or `focal_mean()` that use
`map_overlap`.  But if the new chunk size is not an exact multiple of the
original, dask has to split and recombine blocks — essentially a shuffle —
which tanks performance.

`rechunk_no_shuffle` picks the largest whole-chunk multiple that fits your
target size, so dask can merge blocks in place with zero shuffle overhead.

In [ ]:
import numpy as np
import dask.array as da
import xarray as xr
import xrspatial
from xrspatial.utils import rechunk_no_shuffle

## Create a synthetic dask-backed raster

Start with a 4096 x 4096 raster chunked at 256 x 256 (about 0.25 MB per
chunk for float32).

In [ ]:
np.random.seed(42)
raw = np.random.rand(4096, 4096).astype(np.float32) * 1000
dem = xr.DataArray(
    da.from_array(raw, chunks=256),
    dims=['y', 'x'],
    coords={
        'y': np.linspace(40.0, 41.0, 4096),
        'x': np.linspace(-105.0, -104.0, 4096),
    },
)
print(f'Original chunks: {dem.chunks}')
print(f'Chunks per axis:  {len(dem.chunks[0])} x {len(dem.chunks[1])}')

## Rechunk to ~64 MB target

Each new chunk will be an exact multiple of 256, so dask just groups
existing blocks together.

In [ ]:
big = rechunk_no_shuffle(dem, target_mb=64)
print(f'New chunks:      {big.chunks}')
print(f'Chunks per axis: {len(big.chunks[0])} x {len(big.chunks[1])}')
print(f'Block size:      {big.chunks[0][0]} x {big.chunks[1][0]}')
print(f'Multiple of 256: {big.chunks[0][0] // 256}x')

## Using the .xrs accessor

The same function is available directly on any DataArray.

In [ ]:
big_via_accessor = dem.xrs.rechunk_no_shuffle(target_mb=64)
print(f'Accessor chunks: {big_via_accessor.chunks}')
assert big.chunks == big_via_accessor.chunks

## Compare task graph sizes

Fewer, larger chunks means a smaller task graph for downstream operations.

In [ ]:
from xrspatial.slope import slope

slope_small = slope(dem)
slope_big   = slope(big)

print(f'slope() graph with original chunks: {len(dict(slope_small.data.__dask_graph__())):,} tasks')
print(f'slope() graph with rechunked:       {len(dict(slope_big.data.__dask_graph__())):,} tasks')

## Non-dask arrays pass through unchanged

If the input is a plain numpy-backed DataArray, the function returns it
as-is — no copy, no error.

In [ ]:
numpy_dem = xr.DataArray(raw, dims=['y', 'x'])
result = rechunk_no_shuffle(numpy_dem, target_mb=64)
assert result is numpy_dem
print('Numpy passthrough: OK')

## Dataset support

`rechunk_no_shuffle` also accepts `xr.Dataset`. Each dask-backed variable
is rechunked independently; numpy-backed variables pass through unchanged.

In [ ]:
ds = xr.Dataset({
    'elevation': dem,
    'slope': xr.DataArray(
        da.from_array(np.random.rand(4096, 4096).astype(np.float32), chunks=256),
        dims=['y', 'x'],
    ),
    'mask': xr.DataArray(np.ones((4096, 4096), dtype=np.uint8), dims=['y', 'x']),
})

ds_big = rechunk_no_shuffle(ds, target_mb=64)

for name in ds.data_vars:
    if hasattr(ds[name].data, 'dask'):
        print(f'{name}: {ds[name].chunks[0][0]} -> {ds_big[name].chunks[0][0]}')
    else:
        print(f'{name}: numpy (unchanged)')

In [ ]:
# Works on the Dataset accessor too
ds_big_acc = ds.xrs.rechunk_no_shuffle(target_mb=64)
assert ds_big['elevation'].chunks == ds_big_acc['elevation'].chunks
print('Dataset accessor: OK')